# 🥈 8조 팔로리안 — 난임 환자 임신 성공 예측 v9

> **Public LB 0.7422061 (2등 달성)** — v3 베이스라인 대비 +0.000192

---

## 📌 이 노트북이 하는 일

**6개의 서로 다른 모델을 학습**하고 **가중평균 앙상블**로 합쳐 최종 제출 파일을 만듭니다.

```
[6개 base 모델]
  ① LGBM v3       (94 features)
  ② LGBM v8 + TE  (97 features) - Target Encoding 추가
  ③ XGB v3-feat   (94 features) - XGBoost로 다양성 추가
  ④ XGB v6        (123 features) - 가이드 추가 피처 + XGBoost
  ⑤ 분리 모델      (94 features) - 이식/비이식 별도 학습
  ⑥ CatBoost ⭐   (94 features) - 진짜 다양성의 핵심
        │
        ▼
[6-way 앙상블] scipy.optimize Nelder-Mead 가중치 탐색
        │
        ▼
[제출 파일] 8조_팔로리안_난임프로젝트_v9.csv (LB 0.74221)
```

## ⏱️ 예상 시간

| 단계 | 시간 |
|---|---|
| 환경 설정 + 데이터 로드 | ~2분 |
| ① LGBM v3 | ~14분 |
| ② LGBM v8 (TE) | ~19분 |
| ③ XGB v3-feat | ~11분 |
| ④ XGB v6 | ~12분 |
| ⑤ 분리 모델 | ~14분 |
| ⑥ CatBoost ⭐ | ~28분 |
| ⑦ 6-way 앙상블 | ~1분 |
| **총 합계** | **~100분** |

> **세션이 끊겨도 OK**: 각 모델의 OOF/pred는 Drive에 저장되며, 다시 실행하면 이미 학습된 모델은 자동 스킵.

## 🛡️ Data Leakage 방지

- 범주형: train cat_maps만 수집, test에 transform만 (unseen → NaN)
- 결측 대치 안 함 (트리 native NaN 처리)
- Target Encoding: Fold 내 OOF 방식 + smoothing
- 앙상블 가중치: OOF에서만 scipy.optimize

## 📂 사용 방법

**1. Google Drive에 데이터 업로드**:
```
Google Drive/dacon_난임/data/
  ├── train.csv
  ├── test.csv
  └── sample_submission.csv
```

**2. 위에서 아래로 셀을 순서대로 실행**

**3. 마지막 셀이 `8조_팔로리안_난임프로젝트_v9.csv` 생성** → Drive에서 다운로드 → Dacon 제출

---

## 1️⃣ 환경 설정 — 라이브러리 설치

코랩에 이미 설치되어 있을 수 있지만, **버전 호환성**을 위해 명시적으로 고정합니다.

In [ ]:
# 학습에 필요한 4개 라이브러리 설치
# - lightgbm: ① LGBM v3, ② LGBM v8, ⑤ 분리 모델용
# - xgboost : ③ XGB v3-feat, ④ XGB v6용
# - catboost: ⑥ CatBoost (앙상블 다양성의 핵심)
# - scipy   : ⑦ 앙상블 가중치 최적화 (Nelder-Mead)
!pip install -q lightgbm==4.3.0 xgboost==2.0.3 catboost==1.2.5 scipy

---

## 2️⃣ 라이브러리 import

학습 + 평가 + 메모리 관리에 필요한 모듈을 한 번에 import.

In [ ]:
import os
import gc                                          # 메모리 정리 (가비지 컬렉터)
import time                                        # 학습 시간 측정
import warnings
from pathlib import Path                           # OS 독립적인 파일 경로

import numpy as np
import pandas as pd

# 모델링 라이브러리
from sklearn.model_selection import StratifiedKFold  # 5-fold 분할 (타겟 비율 보존)
from sklearn.metrics import roc_auc_score            # 대회 평가 지표
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier, Pool

# 앙상블 가중치 최적화
from scipy.optimize import minimize

warnings.filterwarnings('ignore')

print('LightGBM:', lgb.__version__)
print('XGBoost :', xgb.__version__)

---

## 3️⃣ Google Drive 마운트

데이터를 Drive에서 읽고, 학습 결과(OOF/pred)를 Drive에 저장합니다.

> **첫 실행 시 인증 팝업이 뜹니다** — Google 계정 선택 후 권한 허용.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# 경로 설정 (Drive 폴더 구조에 맞게 수정 가능)
# DATA_DIR: train.csv, test.csv, sample_submission.csv가 있는 폴더
# OUT_DIR : 학습 결과(OOF, pred, 제출 파일)를 저장할 폴더
DATA_DIR = Path('/content/drive/MyDrive/dacon_난임/data')
OUT_DIR  = Path('/content/drive/MyDrive/dacon_난임/output')
OUT_DIR.mkdir(parents=True, exist_ok=True)        # 폴더 없으면 생성

# 파일 존재 여부 확인 (없으면 즉시 에러)
assert (DATA_DIR/'train.csv').exists(), f'train.csv 가 {DATA_DIR}에 없습니다'
assert (DATA_DIR/'test.csv').exists(), f'test.csv 가 {DATA_DIR}에 없습니다'
assert (DATA_DIR/'sample_submission.csv').exists(), f'sample_submission.csv 가 {DATA_DIR}에 없습니다'

print('데이터 파일 확인 완료')
print(f'결과 저장 위치: {OUT_DIR}')

---

## 4️⃣ 공통 설정 — 상수, 매핑 사전, dtype

도메인 지식 기반의 매핑 사전들을 정의합니다.

- **`age_map`**: 시술 당시 나이를 중위값으로 변환 (`만35-37세 → 36`)
- **`donor_age_map`**: 기증자 나이도 동일 방식
- **`_count_map`**: `'6회 이상'` → 6 (검열 마커는 별도 피처로 처리)
- **`proc_tokens`**: `특정 시술 유형` 토큰 목록 (예: `'IVF + ICSI'` → `has_IVF=1, has_ICSI=1`)
- **`KEEP_CAT_COLS`**: LightGBM에 categorical로 넘길 컬럼 목록

> 도메인: 연령별 IVF 성공률은 20대 35-45% → 43세+ 5% 이하로 **비선형적**. 그래서 단순 숫자가 아닌 중위값 매핑 사용.

In [ ]:
# === 기본 상수 ===
TARGET = '임신 성공 여부'
ID_COL = 'ID'
N_FOLDS = 5

# === Seed 리스트 (배깅 안정화) ===
# LGBM은 빠르므로 5-seed, XGB/CatBoost는 시간 절약 위해 3-seed
SEEDS_5 = [42, 2024, 777, 31415, 99999]
SEEDS_3 = [42, 2024, 777]

# === 도메인 지식 매핑 사전 ===
# 시술 당시 나이 → 중위값 (가이드: 연령별 IVF 성공률 35-45% → 5% 이하)
age_map = {
    '만18-34세': 26,
    '만35-37세': 36,
    '만38-39세': 38.5,
    '만40-42세': 41,
    '만43-44세': 43.5,
    '만45-50세': 47.5,
    '알 수 없음': np.nan,
}

# 기증자 나이 (난자/정자 공통)
donor_age_map = {
    '만20세 이하': 19,
    '만21-25세': 23,
    '만26-30세': 28,
    '만31-35세': 33,
    '만36-40세': 38,
    '만41-45세': 43,
    '알 수 없음': np.nan,
}

# 시술/임신/출산 횟수 매핑: '6회 이상'은 6으로
count_cols = [
    '총 시술 횟수', '클리닉 내 총 시술 횟수', 'IVF 시술 횟수', 'DI 시술 횟수',
    '총 임신 횟수', 'IVF 임신 횟수', 'DI 임신 횟수',
    '총 출산 횟수', 'IVF 출산 횟수', 'DI 출산 횟수',
]
_count_map = {'0회':0.0, '1회':1.0, '2회':2.0, '3회':3.0, '4회':4.0, '5회':5.0, '6회 이상':6.0}

# `특정 시술 유형` 복합 문자열에서 추출할 토큰
# 예: 'IVF + ICSI + BLASTOCYST' → has_IVF=1, has_ICSI=1, has_BLASTOCYST=1
proc_tokens = ['ICSI', 'IVF', 'BLASTOCYST', 'AH', 'Unknown', 'FER']

# `배아 생성 주요 이유` 토큰 (multi-hot)
reasons = ['기증용', '난자 저장용', '배아 저장용', '현재 시술용']

# LightGBM에 categorical로 넘길 컬럼들 (native categorical 처리)
KEEP_CAT_COLS = [
    '시술 시기 코드', '시술 당시 나이', '시술 유형',
    '배란 유도 유형', '난자 출처', '정자 출처',
    '난자 기증자 나이', '정자 기증자 나이',
]

# === dtype 사전 (메모리 효율적인 로딩용) ===
# 200행만 미리 읽어 어떤 컬럼이 object/float인지 판단
probe = pd.read_csv(DATA_DIR/'train.csv', nrows=200)
obj_cols_src = [c for c in probe.select_dtypes(include='object').columns if c != ID_COL]
dtype_dict = {}
for c in probe.columns:
    if c == ID_COL:
        continue
    dtype_dict[c] = 'object' if c in obj_cols_src else 'float32'
del probe

print(f'dtype 사전: {len(dtype_dict)}개 컬럼')
print(f'학습 타겟 : {TARGET}')

---

## 5️⃣ 공통 피처 엔지니어링 — `process_chunk`

청크 단위로 적용되는 공통 변환들. 이 함수가 만드는 피처는 모든 모델이 공유합니다.

### 만드는 피처들

| 그룹 | 피처 수 | 설명 |
|---|---|---|
| 횟수 컬럼 변환 | 10개 | `'6회 이상'` → 6.0 정수화 |
| 나이 수치화 | 3개 | `age_num`, `egg_donor_age_num`, `sperm_donor_age_num` |
| 토큰 추출 (시술 유형) | 6개 | `has_ICSI`, `has_IVF`, `has_BLASTOCYST`, ... |
| 토큰 추출 (배아 이유) | 4개 | `reason_기증용`, `reason_현재 시술용`, ... |
| 비율 파생 | 8개 | `past_preg_rate`, `embryo_transfer_ratio`, `icsi_ratio`, ... |
| 결측 플래그 | 3개 | `__isnan` 컬럼 (결측 패턴이 시그널) |

In [ ]:
def process_chunk(df):
    """공통 FE - 청크 단위 (메모리 효율)"""

    # [1] 횟수 컬럼 변환: '6회 이상' 같은 문자열 → 정수
    # 도메인: 반복 시술 횟수가 많을수록 부정 예측 (Templeton 1996 등)
    for c in count_cols:
        if c in df.columns:
            df[c] = df[c].astype(str).replace('nan', np.nan).map(_count_map).astype('float32')

    # [2] 나이 → 수치 변환
    # 도메인: 연령별 IVF 성공률 (20대 35-45% → 43세+ 5% 이하)
    df['age_num'] = df['시술 당시 나이'].map(age_map).astype('float32')
    df['egg_donor_age_num'] = df['난자 기증자 나이'].map(donor_age_map).astype('float32')
    df['sperm_donor_age_num'] = df['정자 기증자 나이'].map(donor_age_map).astype('float32')

    # [3] 특정 시술 유형 토큰 분해
    # 'IVF + ICSI' 같은 복합 문자열 → has_ICSI=1, has_IVF=1
    s = df['특정 시술 유형'].fillna('').astype(str)
    for tok in proc_tokens:
        df[f'has_{tok}'] = s.str.contains(tok, regex=False).astype('int8')

    # [4] 배아 생성 주요 이유 토큰 (multi-hot)
    s2 = df['배아 생성 주요 이유'].fillna('').astype(str)
    for r in reasons:
        df[f'reason_{r}'] = s2.str.contains(r, regex=False).astype('int8')
    df = df.drop(columns=['배아 생성 주요 이유'])

    # [5] 비율 파생 (절대값보다 비율이 의미 있음)
    eps = 1e-6   # 0 나누기 방지
    df['past_preg_rate']  = (df['총 임신 횟수']  / (df['총 시술 횟수'] + eps)).astype('float32')   # 과거 임신 효율
    df['past_birth_rate'] = (df['총 출산 횟수']  / (df['총 임신 횟수'] + eps)).astype('float32')   # 임신→출산 전환율
    df['ivf_preg_rate']   = (df['IVF 임신 횟수'] / (df['IVF 시술 횟수']+ eps)).astype('float32')   # IVF 한정 효율
    df['embryo_transfer_ratio'] = (df['이식된 배아 수']  / (df['총 생성 배아 수'] + eps)).astype('float32')  # 배아 활용률
    df['embryo_stored_ratio']   = (df['저장된 배아 수']  / (df['총 생성 배아 수'] + eps)).astype('float32')  # 잉여 배아 비율
    df['icsi_ratio']            = (df['미세주입된 난자 수'] / (df['혼합된 난자 수']  + eps)).astype('float32')  # ICSI 사용 비율
    df['age_x_transferred']     = (df['age_num'] * df['이식된 배아 수']).astype('float32')                      # 나이×이식 상호작용
    df['gap_transfer_minus_pickup'] = (df['배아 이식 경과일'] - df['난자 채취 경과일']).astype('float32')          # 채취-이식 간격

    # [6] 결측 플래그 (결측 자체가 정보)
    # 도메인: 결측 패턴이 클리닉 정책을 시그널함
    for c in ['착상 전 유전 검사 사용 여부', '임신 시도 또는 마지막 임신 경과 연수', '배아 해동 경과일']:
        if c in df.columns:
            df[f'{c}__isnan'] = df[c].isna().astype('int8')

    return df


print('process_chunk 함수 정의 완료')

---

## 6️⃣ v3 핵심 도메인 피처 — `add_v3_features`

가이드의 29개 파생변수 중 **임상적으로 가장 중요한 5개만** 선택. **트리 모델이 자동 학습 못 하는 것들만** 추가합니다.

| # | 피처명 | 한국어 의미 | 왜 필요한가 |
|---|---|---|---|
| V3-1 | `days_mix_to_transfer` | 배아 배양 기간 | 3일(분할기) vs 5일(배반포)은 임상 결정 |
| V3-2 | `real_age_num` | 실제 난자 제공자 나이 | **임신 성공의 가장 강한 변수** (자가/기증 분기) |
| V3-3 | `log_n_embryo` | 배아 수의 로그 변환 | 한계효용 체감 (5→6 < 1→2) |
| V3-4 | `miss_gene_test` | 유전검사 결측 통합 | 결측 패턴이 클리닉 정책 시그널 |
| V3-5 | `age_x_embryo_ratio` | 나이 × 이식률 상호작용 | 고령일수록 이식 개수 늘림 |

> **나머지 24개를 보류한 이유**: 가이드 Part 07이 직접 명시 — *"GBM은 원본만 써도 충분, 트리는 상호작용 자동 학습"*. v6에서 직접 검증한 결과 OOF -0.00006, LB -0.00013으로 **떨어짐**.

In [ ]:
def add_v3_features(df):
    """v3 핵심 도메인 피처 5개"""

    # [V3-1] days_mix_to_transfer (배아 배양 기간)
    # 도메인: 3일(분할기) vs 5일(배반포) - 배반포가 성공률 높음
    df['days_mix_to_transfer'] = (df['배아 이식 경과일'] - df['난자 혼합 경과일']).astype('float32')

    # [V3-2] real_age_num (실제 난자 제공자 나이) ⭐ 임신 성공의 가장 강한 변수
    # 자가시술 → 본인 나이, 기증 → 기증자 나이
    # 43세+ 자가시술 환자가 기증으로 회피하는 패턴 포착
    is_donor = (df['난자 출처'].astype(str) == '기증 제공')
    df['real_age_num'] = df['age_num'].astype('float32').values.copy()
    df.loc[is_donor, 'real_age_num'] = df.loc[is_donor, 'egg_donor_age_num']
    df['real_age_num'] = df['real_age_num'].astype('float32')

    # [V3-3] log_n_embryo (배아 수의 로그 변환)
    # 도메인: 한계효용 체감 - 5→6 차이 < 1→2 차이
    # 트리는 비선형 변환 자동으로 못 함 → 명시적 추가 가치
    df['log_n_embryo'] = np.log1p(df['총 생성 배아 수'].fillna(0)).astype('float32')

    # [V3-4] miss_gene_test (유전검사 결측 통합)
    # 도메인: PGS/PGD/착상검사 결측 패턴 = 클리닉 정책 시그널
    # 가이드 분석: 결측 자체가 -28%p 타겟 격차 보임
    df['miss_gene_test'] = (
        df['PGS 시술 여부'].isna() |
        df['PGD 시술 여부'].isna() |
        df['착상 전 유전 검사 사용 여부'].isna()
    ).astype('int8')

    # [V3-5] age_x_embryo_ratio (나이 × 이식률 상호작용)
    # 도메인: 고령일수록 이식 개수 늘림 (의사 판단)
    df['age_x_embryo_ratio'] = (df['age_num'] * df['embryo_transfer_ratio']).astype('float32')

    return df


print('add_v3_features 함수 정의 완료')

---

## 7️⃣ v6 추가 피처 — `add_v6_features` (XGB v6 모델용)

v3 + 가이드 추가 29개 = 총 123 features. **LGBM v6은 LB에서 떨어졌지만**, XGBoost v6는 앙상블 다양성에 기여.

### 추가 피처 6개 그룹

| 그룹 | 개수 | 대표 피처 |
|---|---|---|
| 결측 핵심 플래그 | 4개 | `미싱_배아이식경과일`, `n_missing_row` |
| 시술 이력 | 8개 | `ever_delivered`, `repeated_3plus`, `is_first_attempt` |
| 배아 파이프라인 전환율 | 9개 | `is_FET`, `성숙률`, `ICSI_수정률`, `이식가능률` |
| 시술적합성 | 3개 | `male_factor`, `시술적합성` (4범주) |
| 구간 범주형 | 2개 | `배아수_그룹`, `ICSI_난자_구간` |
| 상호작용 | 3개 | `age_x_procs`, `age_x_log_embryo`, `icsi_fert_x_male` |

In [ ]:
def add_v6_features(df):
    """v6: v3 + 가이드 추가 29개 피처"""
    df = add_v3_features(df)
    eps = 1e-6

    # === Group E: 결측 핵심 플래그 (4개) ===
    # 가이드 분석: 이 컬럼들의 결측 = -28%p 타겟 격차
    df['미싱_배아이식경과일'] = df['배아 이식 경과일'].isna().astype('int8')
    df['미싱_난자채취경과일'] = df['난자 채취 경과일'].isna().astype('int8')
    df['미싱_난자혼합경과일'] = df['난자 혼합 경과일'].isna().astype('int8')
    df['n_missing_row'] = df.isna().sum(axis=1).astype('int16')   # 행별 결측 합계

    # === Group A: 시술 이력 (8개) ===
    # ever_delivered: 가이드 Top 1 긍정 변수 (Templeton 1996)
    df['ever_delivered']   = (df['총 출산 횟수'].fillna(0) > 0).astype('int8')
    df['repeated_3plus']   = (df['총 시술 횟수'].fillna(0) >= 3).astype('int8')        # 반복 실패군
    df['is_first_attempt'] = (df['총 시술 횟수'].fillna(-1) == 0).astype('int8')      # 초회 시도
    df['prev_birth_rate']  = (df['총 출산 횟수'].fillna(0) / (df['총 임신 횟수'].fillna(0) + 1)).astype('float32')
    df['유산_횟수']         = (df['총 임신 횟수'].fillna(0) - df['총 출산 횟수'].fillna(0)).clip(lower=0).astype('float32')
    df['long_infertility'] = (df['임신 시도 또는 마지막 임신 경과 연수'].fillna(0) >= 7).astype('int8')   # Templeton 기준
    df['censored_시술']    = (df['총 시술 횟수'] == 6).astype('int8')                  # '6회 이상' 검열
    df['censored_임신']    = (df['총 임신 횟수'] == 6).astype('int8')

    # === Group B: 배아 파이프라인 전환율 (9개) ===
    # is_FET: 가이드 Top 5 피처 - 동결/신선 주기 구분
    df['is_FET']        = (df['해동된 배아 수'].fillna(0) > 0).astype('int8')
    df['성숙률']         = (df['미세주입된 난자 수'].fillna(0) / (df['수집된 신선 난자 수'].fillna(0) + 1)).astype('float32')
    df['ICSI_수정률']    = (df['미세주입에서 생성된 배아 수'].fillna(0) / (df['미세주입된 난자 수'].fillna(0) + 1)).astype('float32')
    df['이식가능률']     = ((df['이식된 배아 수'].fillna(0) + df['저장된 배아 수'].fillna(0)) / (df['총 생성 배아 수'].fillna(0) + 1)).astype('float32')
    df['이식채택률']     = (df['이식된 배아 수'].fillna(0) / (df['이식된 배아 수'].fillna(0) + df['저장된 배아 수'].fillna(0) + 1)).astype('float32')
    df['is_ICSI']       = (df['미세주입된 난자 수'].fillna(0) > 0).astype('int8')
    df['ICSI_실패수']    = (df['미세주입된 난자 수'].fillna(0) - df['미세주입에서 생성된 배아 수'].fillna(0)).clip(lower=0).astype('float32')
    df['ICSI_비중']      = (df['미세주입된 난자 수'].fillna(0) / (df['총 생성 배아 수'].fillna(0) + 1)).astype('float32')
    df['이식_취소']      = (df['이식된 배아 수'].fillna(-1) == 0).astype('int8')       # freeze-all 전략

    # === Group C: 시술적합성 (남성요인 × ICSI, 3개) ===
    # 가이드 Part 04: ICSI는 남성요인일 때 적절
    male = (
        (df.get('남성 주 불임 원인', 0).fillna(0) == 1) |
        (df.get('남성 부 불임 원인', 0).fillna(0) == 1) |
        (df.get('불임 원인 - 남성 요인', 0).fillna(0) == 1)
    ).astype('int8')
    df['male_factor'] = male

    # 4범주: 남성요인 × ICSI 매트릭스
    suit = np.where(
        (male == 1) & (df['is_ICSI'] == 1), 'AdequateICSI',         # 적절한 ICSI 사용
        np.where(
            (male == 0) & (df['is_ICSI'] == 1), 'OverusedICSI',     # 과사용
            np.where(
                (male == 1) & (df['is_ICSI'] == 0), 'UnderservedICSI',  # 필요한데 안함
                'Neither'
            )
        )
    )
    df['시술적합성'] = suit.astype(object)
    df['ICSI_적합'] = ((male == 1) & (df['is_ICSI'] == 1)).astype('int8')

    # === Group D: 구간 범주형 (2개) ===
    # 배아수_그룹: 가이드 Part 02 - 역U자 패턴 (7-20개가 최적)
    df['배아수_그룹'] = pd.cut(
        df['총 생성 배아 수'], bins=[-0.01, 6, 14, 20, 1000],
        labels=['Poor', 'Normal', 'High', 'Hyper']
    ).astype(object)

    # ICSI 난자 구간: 가이드 Part 05 - sweet spot 10-19개
    df['ICSI_난자_구간'] = pd.cut(
        df['미세주입된 난자 수'], bins=[-0.01, 3, 9, 19, 1000],
        labels=['VeryLow', 'Low', 'Optimal', 'VeryHigh']
    ).astype(object)

    # === Group F: 상호작용 피처 (3개) ===
    df['age_x_procs']      = (df['age_num'].fillna(0) * df['총 시술 횟수'].fillna(0)).astype('float32')   # 나이×시도 누적
    df['age_x_log_embryo'] = (df['age_num'].fillna(0) * df['log_n_embryo']).astype('float32')             # 나이×배아수
    df['icsi_fert_x_male'] = (df['ICSI_수정률'] * df['male_factor']).astype('float32')                    # ICSI효율×남성요인

    return df


# ===== 메모리 효율적인 청크 로딩 =====
def load_chunked(path, chunksize=20000):
    """파일을 20000행씩 잘라 읽으며 process_chunk 적용 → 메모리 절약"""
    parts = []
    for ch in pd.read_csv(path, dtype=dtype_dict, chunksize=chunksize):
        parts.append(process_chunk(ch))
        gc.collect()
    df = pd.concat(parts, ignore_index=True)
    del parts
    gc.collect()
    return df


print('add_v6_features 및 load_chunked 함수 정의 완료')

---

## 8️⃣ 데이터 로드 (train + test)

약 1~2분 소요. **한 번만 로드**하고 모든 모델에서 재사용합니다.

In [ ]:
t0 = time.time()

print('[1] train 로드...')
train_raw = load_chunked(DATA_DIR/'train.csv')
y = train_raw[TARGET].astype(np.int8).values            # 타겟 분리
train_ids = train_raw[ID_COL].values
print(f'  train shape: {train_raw.shape}, target ratio: {y.mean():.4f}')

print('[2] test 로드...')
test_raw = load_chunked(DATA_DIR/'test.csv')
test_ids = test_raw[ID_COL].values
print(f'  test shape: {test_raw.shape}')

print(f'\n총 소요: {time.time()-t0:.1f}s')

---

## 9️⃣ 안전한 범주형 처리 — Data Leakage 방지

> **대회 규정**: train+test 합쳐서 label encoding 금지 (label encoding/one-hot encoding/scaling/get_dummies/결측 처리에 test 사용 금지)

### 우리의 방어

1. train에서만 unique 값 수집 → `cat_maps` 저장
2. test에 적용할 때 `cat_maps`만 사용
3. test에 train에 없던 새 카테고리 → **NaN 처리**

```python
# ❌ 위반 예 (절대 안 함)
combined = pd.concat([train, test])
pd.get_dummies(combined)   # test 정보가 train에 영향
```

In [ ]:
def make_categorical(train, test, cat_cols):
    """Data Leakage 안전한 범주형 처리"""
    cat_maps = {}

    # === train에서만 카테고리 수집 ===
    for c in cat_cols:
        if c not in train.columns:
            continue
        # 'nan', 'None' 문자열을 진짜 NaN으로
        train[c] = train[c].astype(str).replace({'nan': np.nan, 'None': np.nan})
        vals = sorted([v for v in train[c].dropna().unique()])
        cat_maps[c] = vals
        train[c] = pd.Categorical(train[c], categories=vals)

    # === test에는 train의 카테고리만 적용 ===
    for c in cat_cols:
        if c not in test.columns:
            continue
        test[c] = test[c].astype(str).replace({'nan': np.nan, 'None': np.nan})
        vals = cat_maps[c]
        # train에 없던 새 카테고리 → NaN (누수 방지!)
        unseen = (~test[c].isin(vals)) & test[c].notna()
        if unseen.any():
            test.loc[unseen, c] = np.nan
        test[c] = pd.Categorical(test[c], categories=vals)

    return train, test, cat_maps


print('make_categorical 함수 정의 완료')

---

## 🔟 모델 1/6 — LightGBM v3 (94 features)

**LB 0.74201 검증된 베이스 모델**. 5-seed × 5-fold = 25 fold 학습. 약 14분 소요.

### 하이퍼파라미터
| 파라미터 | 값 | 의미 |
|---|---|---|
| `learning_rate` | 0.025 | 작게 → 안정적 |
| `num_leaves` | 95 | leaf-wise growth |
| `min_data_in_leaf` | 150 | 리프당 최소 샘플 |
| `feature_fraction` | 0.8 | 컬럼 80% sampling |
| `bagging_fraction` | 0.85 | 행 85% sampling |

> **재시작 가능**: 이미 학습한 OOF/pred가 Drive에 있으면 자동 스킵.

In [ ]:
# 모델 1 학습 (이미 학습됐으면 스킵 - 재시작 가능)
if (OUT_DIR/'oof_v3.npy').exists() and (OUT_DIR/'pred_v3.npy').exists():
    print('[skip] LGBM v3 이미 학습됨 - 파일 로드')
    oof_v3 = np.load(OUT_DIR/'oof_v3.npy')
    pred_v3 = np.load(OUT_DIR/'pred_v3.npy')
    print(f'  loaded OOF AUC = {roc_auc_score(y, oof_v3):.5f}')
else:
    t0 = time.time()

    # === 데이터 준비 ===
    train = train_raw.drop(columns=[TARGET, ID_COL]).copy()
    test  = test_raw.drop(columns=[ID_COL]).copy()
    train = add_v3_features(train)               # v3 도메인 피처 5개 추가
    test  = add_v3_features(test)

    # 범주형 처리 (누수 방지)
    cat_cols = [c for c in KEEP_CAT_COLS if c in train.columns]
    train, test, _ = make_categorical(train, test, cat_cols)
    print(f'  features: {len(train.columns)} (v3=94 기대)')

    # === 하이퍼파라미터 ===
    base_params = dict(
        objective='binary', metric='auc',
        learning_rate=0.025,
        num_leaves=95, max_depth=-1,
        feature_fraction=0.8,
        bagging_fraction=0.85, bagging_freq=1,
        min_data_in_leaf=150,
        lambda_l1=0.05, lambda_l2=1.0,
        verbose=-1, n_jobs=-1,
    )

    oof_v3  = np.zeros(len(train), dtype=np.float32)
    pred_v3 = np.zeros(len(test),  dtype=np.float32)

    # === 5-seed 배깅 ===
    for sd in SEEDS_5:
        skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=sd)
        oof_seed  = np.zeros(len(train), dtype=np.float32)
        pred_seed = np.zeros(len(test),  dtype=np.float32)

        for fold, (tr_idx, va_idx) in enumerate(skf.split(train, y)):
            t_f = time.time()
            params = {**base_params,
                      'random_state': sd, 'seed': sd,
                      'bagging_seed': sd, 'feature_fraction_seed': sd}

            dtr = lgb.Dataset(train.iloc[tr_idx], label=y[tr_idx],
                              categorical_feature=cat_cols, free_raw_data=True)
            dva = lgb.Dataset(train.iloc[va_idx], label=y[va_idx],
                              categorical_feature=cat_cols, reference=dtr, free_raw_data=True)

            model = lgb.train(params, dtr, num_boost_round=5000,
                              valid_sets=[dva], valid_names=['v'],
                              callbacks=[lgb.early_stopping(120), lgb.log_evaluation(0)])

            # OOF: validation fold만 예측
            oof_seed[va_idx] = model.predict(train.iloc[va_idx], num_iteration=model.best_iteration)
            # test pred: 5-fold 평균
            pred_seed += model.predict(test, num_iteration=model.best_iteration) / N_FOLDS

            print(f'  sd={sd} fold{fold+1} AUC={roc_auc_score(y[va_idx], oof_seed[va_idx]):.5f} t={time.time()-t_f:.1f}s')
            del model, dtr, dva; gc.collect()

        print(f' [seed{sd}] OOF = {roc_auc_score(y, oof_seed):.5f}')
        oof_v3 += oof_seed; pred_v3 += pred_seed
        del oof_seed, pred_seed; gc.collect()

    # 5-seed 평균
    oof_v3  /= len(SEEDS_5)
    pred_v3 /= len(SEEDS_5)

    # Drive에 저장 (재실행 시 스킵 가능)
    np.save(OUT_DIR/'oof_v3.npy',  oof_v3)
    np.save(OUT_DIR/'pred_v3.npy', pred_v3)
    del train, test; gc.collect()

    print(f'\n[FINAL] LGBM v3 OOF = {roc_auc_score(y, oof_v3):.5f}  (기대: 0.74045)')
    print(f'[Done] {time.time()-t0:.1f}s')

---

## 1️⃣1️⃣ 모델 2/6 — LightGBM v8 + Target Encoding (97 features)

**LB 0.74210** (v3보다 미미하게 ↑). TE 3개 피처 추가:

| 피처 | 한국어 의미 | smoothing |
|---|---|---|
| `TE__proc_specific` | 특정 시술 유형 평균 성공률 | 30 |
| `TE__proc_age` | 시술유형 × 나이 조합 평균 | 25 |
| `TE__proc_timing` | 시술유형 × 시기 조합 평균 | 25 |

### 누수 방지 핵심
**Fold 내 OOF 방식**: 각 fold의 train fold에서만 평균 계산 → validation fold에 적용 → test에는 fold별 평균.

In [ ]:
# === Target Encoding 헬퍼 함수들 ===

def add_te_combo_keys(df):
    """TE를 위한 보조 키 컬럼 생성"""
    df['_te_proc_specific'] = df['특정 시술 유형'].astype(str).replace('nan', 'NA')
    df['_te_proc_age']      = df['시술 유형'].astype(str) + '_' + df['시술 당시 나이'].astype(str)
    df['_te_proc_timing']   = df['시술 유형'].astype(str) + '_' + df['시술 시기 코드'].astype(str)
    return df


def smoothed_te(values, y_tr, smoothing=20.0):
    """Smoothing TE 계산 (희귀 카테고리 안정화)
    공식: (그룹평균 × N + 글로벌평균 × smoothing) / (N + smoothing)
    """
    global_mean = float(np.mean(y_tr))
    df_te = pd.DataFrame({'k': values, 'y': y_tr})
    agg = df_te.groupby('k', observed=True)['y'].agg(['mean', 'count'])
    smoothed = (agg['mean'] * agg['count'] + global_mean * smoothing) / (agg['count'] + smoothing)
    return smoothed.to_dict(), global_mean


def build_te_features(train, test, y, te_cols, sm_map, n_folds=5, te_seed=42):
    """누수 없는 TE 생성:
    - train: 각 fold의 train fold에서만 mapping 계산 → va_idx 적용
    - test : 모든 fold의 mapping 평균
    """
    new_cols = {}
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=te_seed)

    for col in te_cols:
        sm = sm_map.get(col, 20.0)
        oof = np.zeros(len(train), dtype=np.float32)
        test_pred = np.zeros(len(test), dtype=np.float32)

        for fold, (tr_idx, va_idx) in enumerate(skf.split(train, y)):
            mapping, gmean = smoothed_te(train[col].iloc[tr_idx].values,
                                          y[tr_idx], smoothing=sm)
            oof[va_idx]  = train[col].iloc[va_idx].map(mapping).fillna(gmean).values
            test_pred   += test[col].map(mapping).fillna(gmean).values / n_folds

        nm = f'TE__{col[4:] if col.startswith("_te_") else col}'
        new_cols[nm] = (oof, test_pred)

    return new_cols


# === 모델 2 학습 ===
if (OUT_DIR/'oof_v8.npy').exists() and (OUT_DIR/'pred_v8.npy').exists():
    print('[skip] LGBM v8 이미 학습됨')
    oof_v8 = np.load(OUT_DIR/'oof_v8.npy')
    pred_v8 = np.load(OUT_DIR/'pred_v8.npy')
    print(f'  loaded OOF AUC = {roc_auc_score(y, oof_v8):.5f}')
else:
    t0 = time.time()

    train = train_raw.drop(columns=[TARGET, ID_COL]).copy()
    test  = test_raw.drop(columns=[ID_COL]).copy()
    train = add_v3_features(train)
    test  = add_v3_features(test)
    train = add_te_combo_keys(train)
    test  = add_te_combo_keys(test)

    # === Target Encoding (Fold 내 OOF 방식) ===
    print('TE 계산 중...')
    te_cols = ['_te_proc_specific', '_te_proc_age', '_te_proc_timing']
    sm_map = {
        '_te_proc_specific': 30.0,    # 24범주, 일부 희귀 → 강한 smoothing
        '_te_proc_age': 25.0,
        '_te_proc_timing': 25.0,
    }
    te_features = build_te_features(train, test, y, te_cols, sm_map, n_folds=N_FOLDS, te_seed=42)

    for nm, (oof_te, pred_te) in te_features.items():
        train[nm] = oof_te
        test[nm]  = pred_te

    # 보조 컬럼 제거
    train = train.drop(columns=te_cols + ['특정 시술 유형'])
    test  = test.drop(columns=te_cols + ['특정 시술 유형'])

    cat_cols = [c for c in KEEP_CAT_COLS if c in train.columns]
    train, test, _ = make_categorical(train, test, cat_cols)
    print(f'  features: {len(train.columns)} (v8=97 기대)')

    # 하이퍼파라미터 (v3와 동일)
    base_params = dict(
        objective='binary', metric='auc',
        learning_rate=0.025, num_leaves=95, max_depth=-1,
        feature_fraction=0.8, bagging_fraction=0.85, bagging_freq=1,
        min_data_in_leaf=150, lambda_l1=0.05, lambda_l2=1.0,
        verbose=-1, n_jobs=-1,
    )

    oof_v8  = np.zeros(len(train), dtype=np.float32)
    pred_v8 = np.zeros(len(test),  dtype=np.float32)

    for sd in SEEDS_5:
        skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=sd)
        oof_seed  = np.zeros(len(train), dtype=np.float32)
        pred_seed = np.zeros(len(test),  dtype=np.float32)

        for fold, (tr_idx, va_idx) in enumerate(skf.split(train, y)):
            t_f = time.time()
            params = {**base_params,
                      'random_state': sd, 'seed': sd,
                      'bagging_seed': sd, 'feature_fraction_seed': sd}
            dtr = lgb.Dataset(train.iloc[tr_idx], label=y[tr_idx],
                              categorical_feature=cat_cols, free_raw_data=True)
            dva = lgb.Dataset(train.iloc[va_idx], label=y[va_idx],
                              categorical_feature=cat_cols, reference=dtr, free_raw_data=True)
            model = lgb.train(params, dtr, num_boost_round=5000,
                              valid_sets=[dva], valid_names=['v'],
                              callbacks=[lgb.early_stopping(120), lgb.log_evaluation(0)])
            oof_seed[va_idx] = model.predict(train.iloc[va_idx], num_iteration=model.best_iteration)
            pred_seed += model.predict(test, num_iteration=model.best_iteration) / N_FOLDS
            print(f'  sd={sd} fold{fold+1} AUC={roc_auc_score(y[va_idx], oof_seed[va_idx]):.5f} t={time.time()-t_f:.1f}s')
            del model, dtr, dva; gc.collect()
        print(f' [seed{sd}] OOF = {roc_auc_score(y, oof_seed):.5f}')
        oof_v8 += oof_seed; pred_v8 += pred_seed
        del oof_seed, pred_seed; gc.collect()

    oof_v8 /= len(SEEDS_5); pred_v8 /= len(SEEDS_5)

    np.save(OUT_DIR/'oof_v8.npy',  oof_v8)
    np.save(OUT_DIR/'pred_v8.npy', pred_v8)
    del train, test; gc.collect()

    print(f'\n[FINAL] LGBM v8 OOF = {roc_auc_score(y, oof_v8):.5f}  (기대: 0.74047)')
    print(f'[Done] {time.time()-t0:.1f}s')

---

## 1️⃣2️⃣ 모델 3/6 — XGBoost v3-feat (94 features)

v3와 같은 피처셋이지만 **다른 알고리즘** (XGBoost: level-wise vs LGBM: leaf-wise).
3-seed × 5-fold. 약 11분.

In [ ]:
if (OUT_DIR/'oof_xgb_v3f.npy').exists() and (OUT_DIR/'pred_xgb_v3f.npy').exists():
    print('[skip] XGB v3-feat 이미 학습됨')
    oof_xgb_v3f = np.load(OUT_DIR/'oof_xgb_v3f.npy')
    pred_xgb_v3f = np.load(OUT_DIR/'pred_xgb_v3f.npy')
    print(f'  loaded OOF AUC = {roc_auc_score(y, oof_xgb_v3f):.5f}')
else:
    t0 = time.time()

    train = train_raw.drop(columns=[TARGET, ID_COL]).copy()
    test  = test_raw.drop(columns=[ID_COL]).copy()
    train = add_v3_features(train)
    test  = add_v3_features(test)

    cat_cols = [c for c in KEEP_CAT_COLS if c in train.columns]
    train, test, _ = make_categorical(train, test, cat_cols)

    # XGBoost는 메모리 효율을 위해 float32로 다운캐스트
    for c in train.select_dtypes(include='float64').columns:
        train[c] = train[c].astype('float32')
        test[c]  = test[c].astype('float32')

    print(f'  features: {len(train.columns)}')

    XGB_PARAMS = dict(
        objective='binary:logistic', eval_metric='auc',
        learning_rate=0.03, max_depth=7,         # level-wise 트리 깊이
        min_child_weight=10,
        subsample=0.85, colsample_bytree=0.8,
        reg_alpha=0.1, reg_lambda=1.0,
        tree_method='hist', device='cpu',         # histogram 알고리즘 (빠름)
        n_jobs=-1, verbosity=0,
    )

    oof_xgb_v3f  = np.zeros(len(train), dtype=np.float32)
    pred_xgb_v3f = np.zeros(len(test),  dtype=np.float32)

    for sd in SEEDS_3:
        XGB_PARAMS['seed'] = sd
        skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=sd)
        oof_seed  = np.zeros(len(train), dtype=np.float32)
        pred_seed = np.zeros(len(test),  dtype=np.float32)

        for fold, (tr_idx, va_idx) in enumerate(skf.split(train, y)):
            t_f = time.time()
            # XGBoost는 DMatrix로 변환 (categorical 지원)
            dtr = xgb.DMatrix(train.iloc[tr_idx], label=y[tr_idx], enable_categorical=True)
            dva = xgb.DMatrix(train.iloc[va_idx], label=y[va_idx], enable_categorical=True)
            model = xgb.train(XGB_PARAMS, dtr, num_boost_round=3000,
                              evals=[(dva, 'v')], early_stopping_rounds=100, verbose_eval=False)
            oof_seed[va_idx] = model.predict(dva)
            dte = xgb.DMatrix(test, enable_categorical=True)
            pred_seed += model.predict(dte) / N_FOLDS
            print(f'  sd={sd} fold{fold+1} AUC={roc_auc_score(y[va_idx], oof_seed[va_idx]):.5f} t={time.time()-t_f:.1f}s')
            del model, dtr, dva, dte; gc.collect()

        print(f' [seed{sd}] OOF = {roc_auc_score(y, oof_seed):.5f}')
        oof_xgb_v3f += oof_seed; pred_xgb_v3f += pred_seed
        del oof_seed, pred_seed; gc.collect()

    oof_xgb_v3f /= len(SEEDS_3); pred_xgb_v3f /= len(SEEDS_3)
    np.save(OUT_DIR/'oof_xgb_v3f.npy',  oof_xgb_v3f)
    np.save(OUT_DIR/'pred_xgb_v3f.npy', pred_xgb_v3f)
    del train, test; gc.collect()

    print(f'\n[FINAL] XGB v3-feat OOF = {roc_auc_score(y, oof_xgb_v3f):.5f}  (기대: 0.74048)')
    print(f'[Done] {time.time()-t0:.1f}s')

---

## 1️⃣3️⃣ 모델 4/6 — XGBoost v6 (123 features)

가이드 추가 29개 피처 포함. **LGBM v6은 LB에서 떨어졌지만**, XGBoost v6는 앙상블에서 weight 25.8%로 기여.
3-seed × 5-fold. 약 12분.

In [ ]:
if (OUT_DIR/'oof_xgb_v6.npy').exists() and (OUT_DIR/'pred_xgb_v6.npy').exists():
    print('[skip] XGB v6 이미 학습됨')
    oof_xgb_v6 = np.load(OUT_DIR/'oof_xgb_v6.npy')
    pred_xgb_v6 = np.load(OUT_DIR/'pred_xgb_v6.npy')
    print(f'  loaded OOF AUC = {roc_auc_score(y, oof_xgb_v6):.5f}')
else:
    t0 = time.time()

    # v6 피처 사용 = 가이드 +29
    train = train_raw.drop(columns=[TARGET, ID_COL]).copy()
    test  = test_raw.drop(columns=[ID_COL]).copy()
    train = add_v6_features(train)
    test  = add_v6_features(test)

    # v6에서 추가된 범주형 컬럼들
    KEEP_CAT_V6 = ['시술적합성', '배아수_그룹', 'ICSI_난자_구간']
    cat_cols = [c for c in (KEEP_CAT_COLS + KEEP_CAT_V6) if c in train.columns]
    train, test, _ = make_categorical(train, test, cat_cols)

    for c in train.select_dtypes(include='float64').columns:
        train[c] = train[c].astype('float32')
        test[c]  = test[c].astype('float32')

    print(f'  features: {len(train.columns)} (v6=123 기대)')

    XGB_PARAMS = dict(
        objective='binary:logistic', eval_metric='auc',
        learning_rate=0.03, max_depth=7,
        min_child_weight=10,
        subsample=0.85, colsample_bytree=0.8,
        reg_alpha=0.1, reg_lambda=1.0,
        tree_method='hist', device='cpu',
        n_jobs=-1, verbosity=0,
    )

    oof_xgb_v6  = np.zeros(len(train), dtype=np.float32)
    pred_xgb_v6 = np.zeros(len(test),  dtype=np.float32)

    for sd in SEEDS_3:
        XGB_PARAMS['seed'] = sd
        skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=sd)
        oof_seed  = np.zeros(len(train), dtype=np.float32)
        pred_seed = np.zeros(len(test),  dtype=np.float32)

        for fold, (tr_idx, va_idx) in enumerate(skf.split(train, y)):
            t_f = time.time()
            dtr = xgb.DMatrix(train.iloc[tr_idx], label=y[tr_idx], enable_categorical=True)
            dva = xgb.DMatrix(train.iloc[va_idx], label=y[va_idx], enable_categorical=True)
            model = xgb.train(XGB_PARAMS, dtr, num_boost_round=3000,
                              evals=[(dva, 'v')], early_stopping_rounds=100, verbose_eval=False)
            oof_seed[va_idx] = model.predict(dva)
            dte = xgb.DMatrix(test, enable_categorical=True)
            pred_seed += model.predict(dte) / N_FOLDS
            print(f'  sd={sd} fold{fold+1} AUC={roc_auc_score(y[va_idx], oof_seed[va_idx]):.5f} t={time.time()-t_f:.1f}s')
            del model, dtr, dva, dte; gc.collect()

        print(f' [seed{sd}] OOF = {roc_auc_score(y, oof_seed):.5f}')
        oof_xgb_v6 += oof_seed; pred_xgb_v6 += pred_seed
        del oof_seed, pred_seed; gc.collect()

    oof_xgb_v6 /= len(SEEDS_3); pred_xgb_v6 /= len(SEEDS_3)
    np.save(OUT_DIR/'oof_xgb_v6.npy',  oof_xgb_v6)
    np.save(OUT_DIR/'pred_xgb_v6.npy', pred_xgb_v6)
    del train, test; gc.collect()

    print(f'\n[FINAL] XGB v6 OOF = {roc_auc_score(y, oof_xgb_v6):.5f}  (기대: 0.74052)')
    print(f'[Done] {time.time()-t0:.1f}s')

---

## 1️⃣4️⃣ 모델 5/6 — 분리 모델 (이식/비이식 별도)

### 데이터 분포 인사이트
- 이식 안 함 (16.71%): 성공률 **1.96%** (거의 다 실패)
- 이식 함 (83.29%): 성공률 **30.62%**

격차 28%p이므로 **분리 학습** 시도. 단독 OOF는 v3보다 낮지만, 앙상블 다양성에 weight 10.3%로 기여.
약 14분 소요.

In [ ]:
if (OUT_DIR/'oof_split.npy').exists() and (OUT_DIR/'pred_split.npy').exists():
    print('[skip] 분리 모델 이미 학습됨')
    oof_split = np.load(OUT_DIR/'oof_split.npy')
    pred_split = np.load(OUT_DIR/'pred_split.npy')
    print(f'  loaded OOF AUC = {roc_auc_score(y, oof_split):.5f}')
else:
    t0 = time.time()

    train = train_raw.drop(columns=[TARGET, ID_COL]).copy()
    test  = test_raw.drop(columns=[ID_COL]).copy()
    train = add_v3_features(train)
    test  = add_v3_features(test)

    # === 그룹 분리: 이식 여부 ===
    train_transfer = (train['이식된 배아 수'].fillna(0) > 0).values
    test_transfer  = (test['이식된 배아 수'].fillna(0) > 0).values
    print(f'  Train Transfer  : {train_transfer.sum()}/{len(train_transfer)} ({y[train_transfer].mean()*100:.2f}% 성공)')
    print(f'  Train NoTransfer: {(~train_transfer).sum()}/{len(train_transfer)} ({y[~train_transfer].mean()*100:.2f}% 성공)')

    cat_cols = [c for c in KEEP_CAT_COLS if c in train.columns]
    train, test, _ = make_categorical(train, test, cat_cols)

    base_params = dict(
        objective='binary', metric='auc',
        learning_rate=0.025, num_leaves=95, max_depth=-1,
        feature_fraction=0.8, bagging_fraction=0.85, bagging_freq=1,
        lambda_l1=0.05, lambda_l2=1.0,
        verbose=-1, n_jobs=-1,
    )

    oof_split  = np.zeros(len(train), dtype=np.float32)
    pred_split = np.zeros(len(test),  dtype=np.float32)

    # === 두 그룹 각각 학습 ===
    # min_data_in_leaf을 그룹 크기에 맞춰 조정
    for group_name, train_mask, test_mask, mil in [
        ('NoTransfer', ~train_transfer, ~test_transfer, 100),
        ('Transfer',    train_transfer,  test_transfer, 200),
    ]:
        print(f'\n=== Group: {group_name} ===')
        sub_train = train[train_mask].reset_index(drop=True)
        sub_y     = y[train_mask]
        sub_test  = test[test_mask].reset_index(drop=True)
        params_grp = {**base_params, 'min_data_in_leaf': mil}

        oof_grp  = np.zeros(len(sub_train), dtype=np.float32)
        pred_grp = np.zeros(len(sub_test),  dtype=np.float32)

        for sd in SEEDS_5:
            skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=sd)
            oof_seed  = np.zeros(len(sub_train), dtype=np.float32)
            pred_seed = np.zeros(len(sub_test),  dtype=np.float32)

            for fold, (tr_idx, va_idx) in enumerate(skf.split(sub_train, sub_y)):
                t_f = time.time()
                params = {**params_grp,
                          'random_state': sd, 'seed': sd,
                          'bagging_seed': sd, 'feature_fraction_seed': sd}
                dtr = lgb.Dataset(sub_train.iloc[tr_idx], label=sub_y[tr_idx],
                                  categorical_feature=cat_cols, free_raw_data=True)
                dva = lgb.Dataset(sub_train.iloc[va_idx], label=sub_y[va_idx],
                                  categorical_feature=cat_cols, reference=dtr, free_raw_data=True)
                model = lgb.train(params, dtr, num_boost_round=5000,
                                  valid_sets=[dva], valid_names=['v'],
                                  callbacks=[lgb.early_stopping(120), lgb.log_evaluation(0)])
                oof_seed[va_idx] = model.predict(sub_train.iloc[va_idx], num_iteration=model.best_iteration)
                pred_seed += model.predict(sub_test, num_iteration=model.best_iteration) / N_FOLDS
                print(f'  [{group_name}] sd={sd} fold{fold+1} t={time.time()-t_f:.1f}s')
                del model, dtr, dva; gc.collect()
            oof_grp += oof_seed; pred_grp += pred_seed
            del oof_seed, pred_seed; gc.collect()

        oof_grp  /= len(SEEDS_5)
        pred_grp /= len(SEEDS_5)

        # 그룹별 결과를 전체 배열에 매핑
        oof_split[train_mask] = oof_grp
        pred_split[test_mask] = pred_grp

    np.save(OUT_DIR/'oof_split.npy',  oof_split)
    np.save(OUT_DIR/'pred_split.npy', pred_split)
    del train, test; gc.collect()

    print(f'\n[FINAL] 분리 모델 OOF = {roc_auc_score(y, oof_split):.5f}  (기대: 0.74025)')
    print(f'[Done] {time.time()-t0:.1f}s')

---

## 1️⃣5️⃣ 모델 6/6 — CatBoost ⭐ (94 features) — **핵심 다양성**

### 왜 CatBoost가 핵심인가?

| 알고리즘 | 트리 구조 | 분기 결정 | Categorical 처리 |
|---|---|---|---|
| LightGBM | Leaf-wise (비대칭) | Histogram | 정수 매핑 |
| XGBoost | Level-wise (대칭) | Histogram | 원-핫 또는 정수 |
| **CatBoost** | **Oblivious tree** (완전 대칭) | **Ordered statistics** | **Ordered Target Statistics** |

LGBM/XGBoost는 같은 패밀리 → 비슷한 실수 패턴 (상관 0.9964). CatBoost는 **상관 0.9916~0.9932**로 진짜 다양성.

### 메모리 안전 설정 (이전 OOM 경험에서 학습)
- `border_count=64` (256→64로 양자화 줄임)
- `max_ctr_complexity=0` (CTR 비활성화)
- `subsample=0.7, rsm=0.7`

3-seed × 5-fold. **약 28분** (가장 오래 걸림).

In [ ]:
if (OUT_DIR/'oof_catboost.npy').exists() and (OUT_DIR/'pred_catboost.npy').exists():
    print('[skip] CatBoost 이미 학습됨')
    oof_catboost = np.load(OUT_DIR/'oof_catboost.npy')
    pred_catboost = np.load(OUT_DIR/'pred_catboost.npy')
    print(f'  loaded OOF AUC = {roc_auc_score(y, oof_catboost):.5f}')
else:
    t0 = time.time()

    train = train_raw.drop(columns=[TARGET, ID_COL]).copy()
    test  = test_raw.drop(columns=[ID_COL]).copy()
    train = add_v3_features(train)
    test  = add_v3_features(test)

    # CatBoost는 categorical을 string으로 받음 (NaN 포함)
    cat_cols = [c for c in KEEP_CAT_COLS if c in train.columns]
    for c in cat_cols:
        train[c] = train[c].astype(object).where(train[c].notna(), 'NaN').astype(str)
        test[c]  = test[c].astype(object).where(test[c].notna(), 'NaN').astype(str)

    # float64 → float32 (메모리 절약)
    for c in train.select_dtypes(include='float64').columns:
        train[c] = train[c].astype('float32')
        test[c]  = test[c].astype('float32')

    features = train.columns.tolist()
    cat_idx = [features.index(c) for c in cat_cols]
    print(f'  features: {len(features)}, cat indices: {len(cat_idx)}')
    print(f'  메모리: {train.memory_usage(deep=True).sum()/1e6:.1f} MB')

    # === CatBoost 메모리 안전 하이퍼파라미터 ===
    CAT_PARAMS = dict(
        iterations=600,                  # 600개 트리
        learning_rate=0.05,
        depth=6,                         # 6 (보수적)
        l2_leaf_reg=3.0,
        loss_function='Logloss',
        eval_metric='AUC',
        early_stopping_rounds=60,
        verbose=0,
        task_type='CPU',
        bootstrap_type='Bernoulli',
        subsample=0.7,                   # 행 70% sampling
        rsm=0.7,                          # column 70% sampling
        max_ctr_complexity=0,            # CTR 비활성화 (메모리 절약)
        border_count=64,                  # 양자화 64 단계 (256→64)
        thread_count=-1,
    )

    oof_catboost  = np.zeros(len(train), dtype=np.float32)
    pred_catboost = np.zeros(len(test),  dtype=np.float32)

    for sd in SEEDS_3:
        CAT_PARAMS['random_seed'] = sd
        skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=sd)
        oof_seed  = np.zeros(len(train), dtype=np.float32)
        pred_seed = np.zeros(len(test),  dtype=np.float32)

        for fold, (tr_idx, va_idx) in enumerate(skf.split(train, y)):
            t_f = time.time()

            # CatBoost는 Pool 객체로 학습
            train_pool = Pool(train.iloc[tr_idx], y[tr_idx], cat_features=cat_idx)
            valid_pool = Pool(train.iloc[va_idx], y[va_idx], cat_features=cat_idx)

            model = CatBoostClassifier(**CAT_PARAMS)
            model.fit(train_pool, eval_set=valid_pool, use_best_model=True, verbose=0)

            # predict_proba: [P(0), P(1)] → P(1)만 사용
            oof_seed[va_idx] = model.predict_proba(train.iloc[va_idx])[:, 1]
            pred_seed += model.predict_proba(test)[:, 1] / N_FOLDS

            print(f'  sd={sd} fold{fold+1} AUC={roc_auc_score(y[va_idx], oof_seed[va_idx]):.5f} iter={model.get_best_iteration()} t={time.time()-t_f:.1f}s')
            del model, train_pool, valid_pool; gc.collect()

        print(f' [seed{sd}] OOF = {roc_auc_score(y, oof_seed):.5f}')
        oof_catboost += oof_seed; pred_catboost += pred_seed
        del oof_seed, pred_seed; gc.collect()

    oof_catboost  /= len(SEEDS_3)
    pred_catboost /= len(SEEDS_3)
    np.save(OUT_DIR/'oof_catboost.npy',  oof_catboost)
    np.save(OUT_DIR/'pred_catboost.npy', pred_catboost)
    del train, test; gc.collect()

    print(f'\n[FINAL] CatBoost OOF = {roc_auc_score(y, oof_catboost):.5f}  (기대: 0.74045)')
    print(f'[Done] {time.time()-t0:.1f}s')

---

## 1️⃣6️⃣ 6-way 앙상블 + 최종 제출 ⭐

6개 base 모델의 OOF로 **scipy.optimize Nelder-Mead** 가중치 탐색 → test 예측 가중평균.

### Data Leakage 방지
가중치 탐색에 **OOF만 사용** (test 안 봄). 학습된 가중치를 test 예측에 그대로 적용.

### 예상 결과

| 단계 | OOF | LB |
|---|---|---|
| v3 단독 | 0.74045 | 0.74201 |
| 5-way (CatBoost 없이) | 0.74077 | (미제출) |
| **v9 = 6-way (CatBoost 추가)** | **0.74100** | **0.74221** ⭐ |

In [ ]:
print('=' * 60)
print('6-way 앙상블 가중치 탐색')
print('=' * 60)

# === 6개 base 모델 OOF/pred 모음 ===
oofs = {
    'LGBM_v3':     oof_v3,
    'LGBM_v8_TE':  oof_v8,
    'XGB_v3feat':  oof_xgb_v3f,
    'XGB_v6':      oof_xgb_v6,
    'split_model': oof_split,
    'CatBoost':    oof_catboost,
}
preds = {
    'LGBM_v3':     pred_v3,
    'LGBM_v8_TE':  pred_v8,
    'XGB_v3feat':  pred_xgb_v3f,
    'XGB_v6':      pred_xgb_v6,
    'split_model': pred_split,
    'CatBoost':    pred_catboost,
}

# === 개별 모델 OOF AUC ===
print('\n[개별 OOF AUC]')
for k, v in oofs.items():
    print(f'  {k:15s} {roc_auc_score(y, v):.5f}')

# === CatBoost와 다른 모델 상관 (다양성 확인) ===
# 핵심: CatBoost는 다른 모델과 상관 0.991~0.993으로 낮음 → 진짜 다양성
print('\n[CatBoost vs 다른 모델 상관 - 낮을수록 다양성↑]')
for k in oofs:
    if k == 'CatBoost':
        continue
    c = np.corrcoef(oofs['CatBoost'], oofs[k])[0, 1]
    print(f'  CatBoost vs {k:15s} {c:.4f}')

# === 행렬로 변환 (최적화에 사용) ===
keys = list(oofs.keys())
oof_arr  = np.stack([oofs[k]  for k in keys], axis=1).astype(np.float64)
pred_arr = np.stack([preds[k] for k in keys], axis=1).astype(np.float64)


# === 목적 함수: -AUC (최소화 = AUC 최대화) ===
def neg_auc(w):
    w = np.clip(w, 0, None)            # 음수 방지
    s = w.sum()
    if s < 1e-9:
        return 0
    w = w / s                            # normalize
    return -roc_auc_score(y, oof_arr @ w)


# === 5개 다른 시작점에서 Nelder-Mead 최적화 (robust optimum 확인) ===
starts = [
    [0.0, 0.31, 0.165, 0.387, 0.138, 0.0],   # 5-way 결과 + cat=0
    [0.0, 0.25, 0.15, 0.30, 0.10, 0.20],     # cat 20%
    [0.0, 0.20, 0.15, 0.25, 0.10, 0.30],     # cat 30%
    [0.1, 0.20, 0.15, 0.25, 0.10, 0.20],
    [1/6] * 6,                                # 균등 시작
]

best = (None, 0)
print('\n[가중치 탐색]')
for st in starts:
    res = minimize(neg_auc, np.array(st),
                   method='Nelder-Mead',
                   options={'maxiter': 300, 'xatol': 0.001, 'fatol': 1e-7})
    w = np.clip(res.x, 0, None)
    w = w / w.sum()
    auc = -neg_auc(w)
    print(f'  start {[round(x,2) for x in st]} → OOF={auc:.5f}')
    if auc > best[1]:
        best = (w, auc)

# === 최적 가중치 출력 ===
print(f'\n=== 최적 6-way 가중치 ===')
for k, v in zip(keys, best[0]):
    print(f'  {k:15s} {v:.3f}')
print(f'\n>>> 6-way OOF AUC = {best[1]:.5f}  (기대: 0.74100)')
print(f'>>> v3 단독 대비 OOF +{best[1]-roc_auc_score(y, oofs["LGBM_v3"]):.5f}')

# === 최종 test 예측 (가중평균) ===
final_pred = pred_arr @ best[0]
final_pred = np.clip(final_pred, 0, 1)        # 확률은 [0, 1] 범위

# === 제출 파일 생성 ===
sample = pd.read_csv(DATA_DIR/'sample_submission.csv')
sub_df = pd.DataFrame({'ID': test_ids, 'probability': final_pred})
sample = sample[['ID']].merge(sub_df, on='ID', how='left')   # 원본 ID 순서 유지

out_path = OUT_DIR/'8조_팔로리안_난임프로젝트_v9.csv'
sample.to_csv(out_path, index=False)

print(f'\n[제출 파일 저장] {out_path}')
print(f'분포: mean={sample["probability"].mean():.4f}, std={sample["probability"].std():.4f}')
print(f'\n=== 예상 LB: 0.74221 (2등) ===')
print(f'\nDrive에서 위 파일 다운로드 → Dacon 제출')

---

## 1️⃣7️⃣ 다음에 시도할 만한 개선 방향

v9가 끝이 아닙니다. 더 점수를 올리고 싶다면 시도해볼 만한 카드들 — 솔직한 기대 효과와 함께.

### 🎯 단기 카드 (1~3시간 내)

| 카드 | 기대 OOF Δ | 시간 | 위험 |
|---|---|---|---|
| CatBoost 5-seed로 늘림 (현재 3-seed) | +0.00003~0.00007 | ~35분 | 거의 없음 |
| LightGBM dart boosting 추가 | +0.0001~0.0003 | ~1시간 | 시간 큰 데 비해 효과 미지수 |
| 10-fold CV로 OOF 안정화 | LB 안정성 ↑ (OOF는 비슷) | ~2시간 | 거의 없음 |

### 🚀 중기 카드 (반나절~하루)

| 카드 | 기대 효과 | 위험 |
|---|---|---|
| 정확한 Stacking (fold split 일관성 보장) | OOF +0.0005~0.002 | 구현 복잡, 디버깅 필요 |
| Pseudo-labeling (very confident test → train) | LB +0.001~0.003 | 자기 예측이라 overfitting 위험 큼 |
| 신경망 추가 (TabNet, FT-Transformer) | 진짜 다른 패밀리 → 다양성 ↑ | 시간/메모리 많이 필요, 효과 미지수 |

### 💡 본질적 개선 카드 (난이도 ↑↑)

| 카드 | 설명 |
|---|---|
| 환자 단위 그룹 인식 | 같은 환자가 여러 행에 있다면, GroupKFold로 누수 방지 |
| 시술 시기별 모델 분리 | 시기 코드별로 분포가 다를 가능성 |
| 도메인 의사 자문 | 데이터에 안 보이는 임상 지식 발견 |
| AutoGluon 기반 자동 앙상블 | 모델 풀 자동 탐색 |

### ⚠️ 솔직한 한계 평가

- v3 → v9: LB +0.00019 (v3 0.74201 → v9 0.74221)
- 이번 시도들 종합 결과: **LB 천장 ~0.7425 추정**
- **0.75 도달은 본질적으로 어려움** (데이터 노이즈 천장)

이유:
1. IVF 성공률 자체가 30~40% **확률 게임** (deterministic 신호 한계)
2. OOF +0.00055가 LB +0.00019로만 전이 (전이율 35%)
3. 모든 트리 모델이 비슷한 천장에서 수렴

---

## 📋 최종 결과 요약

In [ ]:
print('=' * 60)
print('=== v9 최종 결과 요약 ===')
print('=' * 60)
print('\n[6개 base 모델 OOF]')
for k, v in oofs.items():
    print(f'  {k:15s} {roc_auc_score(y, v):.5f}')
print(f'\n[6-way 앙상블 OOF] {best[1]:.5f}')
print(f'\n[제출 파일] {out_path}')
print(f'\n[LB 점수 예상] 0.74221 (2등)')
print(f'\n다음 단계: Drive에서 파일 다운로드 → Dacon 제출')